<a href="https://colab.research.google.com/github/Madhuanabala/breast-cancer/blob/unknown-mol-d%26fp/RDkit_unknown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit-pypi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 61.2 MB/s eta 0:00:00


In [2]:
!pip install mordred

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.8/128.8 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.1 MB/s eta 0:00:00
  Created wheel for mordred: filename=mordred-1.2.0-py3-none-any.whl size=176718 sha256=5823dcb996d1153c792f9478f3da98b278a755ff63351e383caff54fee1ca4d3
  Stored in directory: /root/.cache/pip/wheels/8b/30/0b/84e3f6775306e74cf5957ee4d16b10bf3927dcec44cc23d5f2
Successfully built mordred
  Attempting uninstall: networkx
    Found existing installation: networkx 3.4.2
    Uninstalling networkx-3.4.2:
      Successfully uninstalled networkx-3.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatib

In [3]:
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

import pandas as pd
import numpy as np
from tqdm import tqdm
import os

In [4]:
dataset=pd.read_excel('/content/New Microsoft Excel Worksheet (2).xlsx')


In [5]:
df=dataset.dropna(subset=['smiles'])

In [6]:
df

,Compound name,smiles,IMPPAT_id
0,Berberine,COc1c(OC)ccc2c1c[n+]1CCc3c(-c1c2)cc1c(c3)OCO1,IMPHY005665
1,Luteolin,Oc1cc(O)c2c(c1)oc(cc2=O)c1ccc(c(c1)O)O,IMPHY004660
2,gallic acid,OC(=O)c1cc(O)c(c(c1)O)O,IMPHY012021
3,Chlorogenic acid,O=C(O[C@@H]1C[C@@](O)(C[C@H]([C@H]1O)O)C(=O)O)...,IMPHY011844
4,Andrographolide,OC[C@]1(C)[C@H](O)CC[C@@]2([C@@H]1CCC(=C)[C@H]...,IMPHY004978
...,...,...,...
123,Piplartine,COC1=CC(=CC(=C1OC)OC)/C=C/C(=O)N2CCC=CC2=O,IMPHY006342
124,Liquiritigenin,C1[C@H](OC2=C(C1=O)C=CC(=C2)O)C3=CC=C(C=C3)O,IMPHY001869
125,Phorbol,OCC1=C[C@H]2[C@H]3[C@@](C3(C)C)(O)[C@@H]([C@H]...,IMPHY010284
126,Alpha-Amyrin,C[C@@H]1CC[C@]2([C@@H]([C@H]1C)C1=CC[C@H]3[C@@...,IMPHY011619


In [7]:

def RDkit_descriptors(smiles):
    mols = [Chem.MolFromSmiles(i) for i in smiles]
    calc = MoleculeDescriptors.MolecularDescriptorCalculator([x[0] for x in Descriptors._descList])
    desc_names = calc.GetDescriptorNames()

    Mol_descriptors = []
    for mol in mols:
        # add hydrogens to molecules
        mol = Chem.AddHs(mol)
        # Calculate all 200 descriptors for each molecule
        descriptors = calc.CalcDescriptors(mol)
        Mol_descriptors.append(descriptors)
    return Mol_descriptors, desc_names
# Split the SMILES into chunks of 100,000 for faster processing
chunk_size = 100000
chunks = [df[i:i+chunk_size] for i in range(0, len(df), chunk_size)]

total_chunks=len(chunks)
total_time=0

# Check if there is an existing output file
if os.path.isfile('RDkit_descriptors.csv'):
    existing_data = pd.read_csv('RDkit_descriptors.csv', index_col=0)
else:
    existing_data = pd.DataFrame()
    # Calculate descriptors for each chunk and concatenate the results
for i, chunk in enumerate(tqdm(chunks, desc='Processing', total=len(chunks))):
    # Check if this chunk has already been processed
    if len(existing_data) >= len(chunk):
        continue
    # Calculate descriptors for this chunk
    descriptors, desc_names = RDkit_descriptors(chunk['smiles'])
    # Convert the descriptors to a dataframe
    df_with_200_descriptors = pd.DataFrame(descriptors, columns=desc_names,)
    # Add the chunk index as a new column
    df_with_200_descriptors['chunk_index'] = i
    # Append the data to the existing data
    existing_data = pd.concat([existing_data, df_with_200_descriptors], axis=0)
    # Save the data after each chunk
    existing_data.to_csv('RDkit_descriptors.csv')

# Save the final data
existing_data.to_csv('RDkit_descriptors.csv')

Processing: 100%|██████████| 1/1 [00:05<00:00,  5.82s/it]
